# U00 · Bienvenida a Google Colab y Python

> **Para quien nunca ha programado.** Este cuaderno es tu **red de seguridad**. En 15 minutos verás lo justo para abrir, leer y ejecutar los cuadernos del curso **sin miedo**. No hay que memorizar nada.

> ⚠️ Todos los datos son **sintéticos** (inventados). No representan pacientes reales.

**La promesa del curso:** tú aportas el **criterio clínico**; el **código lo escribe la IA** (o ya está escrito y comentado, como aquí). Tu trabajo es **entender qué hace y por qué**, no teclearlo de memoria.


## 1. ¿Qué es esto que estás viendo?

Estás en un **cuaderno** (*notebook*) de **Google Colab**: una página web gratuita que ejecuta **Python** (un lenguaje de programación) en los ordenadores de Google. No instalas nada; todo pasa en el navegador.

Un cuaderno es una lista de **celdas** que se ejecutan **de arriba abajo**. Hay dos tipos:
- **Celdas de texto** (como esta): explican. Están escritas en un formato sencillo llamado *Markdown*.
- **Celdas de código**: contienen Python. Se ejecutan pulsando el botón **▶** de la izquierda, o con **Mayús + Enter**. El resultado aparece justo debajo.

> 💡 **Truco.** Para ejecutar todo el cuaderno de una vez: menú *Entorno de ejecución → Ejecutar todo*.


## 2. Tu primera celda de código

Debajo hay una celda de código muy simple: una **suma**. Ejecútala (▶ o Mayús+Enter) y verás el resultado aparecer debajo. No rompe nada; puedes ejecutarla mil veces.


In [ ]:
2 + 2

¡Ya has ejecutado código! El `4` que aparece es el resultado. Así de inofensivo es esto.

Python también sabe mostrar texto con `print()`:


In [ ]:
print("Hola. Soy Python, y voy a ayudarte a entender tus datos.")

## 3. Lo mínimo de Python (para leerlo sin miedo)

No vas a escribir Python de memoria, pero reconocer cuatro cosas te hará leer cualquier celda con tranquilidad.

**Una variable** es un nombre que guarda un valor. Vamos a calcular el **índice de masa corporal (IMC)** de un paciente, que se define como el peso dividido por la altura al cuadrado:


In [ ]:
peso_kg = 82        # guardamos el peso en una variable
altura_m = 1.75     # y la altura

imc = peso_kg / (altura_m ** 2)   # ** significa "elevado a"
print("IMC:", round(imc, 1))

**Lo que ha pasado:** guardamos dos números, calculamos el IMC con una fórmula y lo mostramos redondeado a un decimal con `round(...)`. Léelo despacio: es aritmética con nombres.


**Una lista** es una colección de valores. Y un **bucle** (`for`) repite algo para cada elemento. Aquí calculamos el IMC de varios pacientes de golpe:


In [ ]:
pesos =   [82, 70, 95, 60]     # cuatro pacientes
alturas = [1.75, 1.68, 1.80, 1.55]

for peso, altura in zip(pesos, alturas):     # recorre los pacientes uno a uno
    imc = peso / (altura ** 2)
    print(f"Peso {peso} kg, altura {altura} m  ->  IMC {imc:.1f}")

**Lo que ha pasado:** el `for` repite el cálculo para cada paciente. No necesitas escribir esto tú; solo reconocer que "esto se hace una vez por cada fila". Eso es el 90 % del código que verás en el curso.


## 4. Las librerías que verás una y otra vez

Python trae "cajas de herramientas" ya hechas, llamadas **librerías**. En el curso usaremos sobre todo:
- **pandas** → trabajar con **tablas** de datos (como una hoja de cálculo con superpoderes).
- **matplotlib / seaborn** → hacer **gráficas**.
- **numpy** → cálculos con números (casi siempre entre bastidores).

Vamos a usar pandas para crear los datos del curso y abrir la tabla de pacientes. La siguiente celda **genera los datos sintéticos** (es la celda que verás al principio de todos los cuadernos).


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

Ahora abrimos la tabla de pacientes y miramos las **primeras filas** con `.head()`. Es lo primero que se hace con cualquier tabla nueva: asomarse a ver qué pinta tiene.


In [ ]:
import pandas as pd

pacientes = pd.read_csv("pacientes.csv")
pacientes.head()

Cada **fila** es un paciente; cada **columna**, un dato suyo (edad, sexo, tensión…). `.head()` muestra solo las cinco primeras para no llenar la pantalla.

Podemos pedir cosas concretas. Por ejemplo, la **edad media** de la cohorte:


In [ ]:
edad_media = pacientes["edad"].mean()
print(f"Edad media: {edad_media:.1f} años")

Y una **gráfica** sencilla: ¿cómo se reparten las edades? Un histograma cuenta cuántos pacientes hay en cada tramo de edad.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.hist(pacientes["edad"], bins=30, color="#00AEC7", edgecolor="white")
plt.xlabel("Edad (años)")
plt.ylabel("Nº de pacientes")
plt.title("Distribución de edades en la cohorte (datos sintéticos)")
plt.show()

**Lo que vemos:** la mayoría de pacientes son adultos de mediana edad y mayores, con menos jóvenes. Con una sola línea de código hemos "visto" a nuestros 20 000 pacientes. Eso es el poder de estas herramientas.


## 5. El atajo que lo cambia todo: pedírselo a la IA

No tienes que recordar cómo se escribe un histograma. Puedes **describir lo que quieres** a un asistente de IA (el de Colab, Claude, u otro) y él escribe el código. Por ejemplo, podrías pedirle:

> *"Tengo un fichero pacientes.csv con una columna edad. Dibuja un histograma de las edades con títulos en español."*

Y obtendrías, más o menos, la celda de arriba. **Tu papel** es saber **qué pedir** y **entender el resultado** —si el histograma tiene sentido clínico—, no memorizar la sintaxis. En la **U10** convertimos esto en un método de trabajo completo.


## 6. Qué hemos aprendido

- Un cuaderno son **celdas** que se ejecutan de arriba abajo; las de código se lanzan con **▶ / Mayús+Enter**.
- Sabes leer (que no escribir de memoria) una **variable**, una **lista**, un **bucle** y una **fórmula** clínica sencilla.
- **pandas** abre tablas (`read_csv`, `.head()`, `.mean()`) y **matplotlib** dibuja. La primera celda de cada cuaderno **genera los datos** solos.
- Y lo más importante: **el código lo puede escribir la IA**; tú pones el criterio.

**Ya estás listo.** Cuando quieras, abre el cuaderno de la unidad que te toque: funcionan igual que este.
